# 02 — Data: NumPy & Pandas

**Fase:** 1 — Fundamenter  
**Estimert tid:** 2–3 timer  
**Forutsetninger:** Notatbok 01 (Python for AI Engineers)

**Hva du bygger:** Et lite dataanalyse-skript som laster inn et datasett, renser det, og beregner statistikk — akkurat som du ville gjort som forberedelse til å trene eller evaluere en ML-modell.

---

## Hvorfor NumPy og Pandas?

Tenk på en embedding (som du lærer om i notatbok 06) som en liste med 1536 tall:  
`[0.12, -0.34, 0.87, ..., 0.03]`

**NumPy** gjør matematikk på slike lister lynraskt — tusenvis av ganger raskere enn vanlige Python-lister.  
**Pandas** gjør det enkelt å jobbe med tabeller (tenk Excel i Python).

Du trenger begge for å forstå og manipulere data i AI-systemer.

In [ ]:
%pip install -q numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"NumPy {np.__version__}, Pandas {pd.__version__}")

---

## Del 1: NumPy — Matematikk på lister

### Hvorfor ikke bare bruke vanlige Python-lister?

```python
# Python-liste: Ikke vektorisert — løkke under panseret
a = [1, 2, 3]
b = [4, 5, 6]
c = [x + y for x, y in zip(a, b)]  # Treg

# NumPy-array: Vektorisert — matematikk på hele arrayen på én gang
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
c = a + b  # Rask — og mye enklere å lese!
```

In [ ]:
# Grunnleggende NumPy-operasjoner
a = np.array([1, 2, 3, 4, 5])
b = np.array([10, 20, 30, 40, 50])

print("Addisjon:  ", a + b)
print("Multiplik.:", a * b)
print("Kvadrat:   ", a ** 2)
print("Gjennomsn.:", a.mean())
print("Std.avvik: ", a.std())

In [ ]:
# Embedding-eksempel: En enkel tekstvektor (forenklet)
# I virkeligheten er embeddings 768 eller 1536 tall — men prinsippet er det samme

embedding_afp     = np.array([0.8, 0.1, 0.3, 0.9, 0.2])  # "AFP pensjon"
embedding_pensjon = np.array([0.7, 0.2, 0.4, 0.8, 0.3])  # "alderspensjon"
embedding_fly     = np.array([0.1, 0.9, 0.7, 0.1, 0.8])  # "fly til Oslo"

# Cosinus-likhet: Mål på hvor like to vektorer er (1.0 = identisk, 0.0 = urelatert)
def cosinus_likhet(v1: np.ndarray, v2: np.ndarray) -> float:
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print(f"AFP vs Pensjon:  {cosinus_likhet(embedding_afp, embedding_pensjon):.3f}  (høy likhet forventet)")
print(f"AFP vs Fly:      {cosinus_likhet(embedding_afp, embedding_fly):.3f}  (lav likhet forventet)")

In [ ]:
# 2D-arrays: Matriser (brukes overalt i ML)
# Tenk på dette som en tabell: 3 dokumenter, 4 egenskaper hver
matrise = np.array([
    [0.8, 0.1, 0.3, 0.9],  # Dokument 1
    [0.7, 0.2, 0.4, 0.8],  # Dokument 2
    [0.1, 0.9, 0.7, 0.1],  # Dokument 3
])

print("Form:", matrise.shape)         # (3, 4) = 3 rader, 4 kolonner
print("Første rad:", matrise[0])      # Dokument 1
print("Første kolonne:", matrise[:, 0])  # Alle dokumenters første egenskap
print("Snitt per dokument:", matrise.mean(axis=1))  # Snitt langs kolonner

---

## Del 2: Pandas — Tabeller i Python

Pandas `DataFrame` er som et Excel-ark i Python. Du vil bruke det til å laste inn, rense og analysere datasett.

In [ ]:
# Lag en enkel DataFrame
data = {
    "dokument_id": ["spk-001", "spk-002", "spk-003", "spk-004", "spk-005"],
    "tittel": ["AFP-guide", "Alderspensjon", "Uførepensjon", "Barnepensjon", "AFP-guide"],
    "ord_antall": [450, 820, 310, None, 450],  # None = mangler data
    "relevans_score": [0.92, 0.87, 0.65, 0.71, 0.92],
    "publisert_år": [2023, 2022, 2023, 2021, 2023],
}

df = pd.DataFrame(data)
print(df)
print(f"\nForm: {df.shape} ({df.shape[0]} rader, {df.shape[1]} kolonner)")

In [ ]:
# Grunnleggende statistikk — kjør alltid dette på et nytt datasett!
print(df.describe())

In [ ]:
# Datarensing: Håndtere manglende verdier
print("Manglende verdier per kolonne:")
print(df.isnull().sum())

# Fyll inn manglende ord_antall med medianen
df["ord_antall"] = df["ord_antall"].fillna(df["ord_antall"].median())
print("\nEtter rensing — ingen manglende verdier:")
print(df.isnull().sum())

In [ ]:
# Filtrering og sortering
høy_relevans = df[df["relevans_score"] >= 0.85].sort_values("relevans_score", ascending=False)
print("Dokumenter med høy relevans:")
print(høy_relevans[["tittel", "relevans_score"]])

In [ ]:
# Fjerne duplikater
print(f"Rader før: {len(df)}")
df_unik = df.drop_duplicates(subset=["tittel", "ord_antall"])
print(f"Rader etter fjerning av duplikater: {len(df_unik)}")

In [ ]:
# Gruppering — nyttig for å forstå datafordelingen
print("Gjennomsnittlig relevans per år:")
print(df.groupby("publisert_år")["relevans_score"].mean().round(3))

---

## Del 3: Enkel visualisering

Visualisering hjelper deg å *forstå* dataene før du sender dem inn i en modell.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Relevans-score fordeling
axes[0].hist(df["relevans_score"], bins=5, color="steelblue", edgecolor="white")
axes[0].set_title("Fordeling av relevans-score")
axes[0].set_xlabel("Relevans-score")
axes[0].set_ylabel("Antall dokumenter")

# Ord per dokument
axes[1].bar(df["dokument_id"], df["ord_antall"], color="coral", edgecolor="white")
axes[1].set_title("Ord per dokument")
axes[1].set_xlabel("Dokument")
axes[1].set_ylabel("Antall ord")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---

## Miniprosjekt: Forbered et datasett for RAG

Du har fått et råt datasett med pensjonsdokumenter. Rens det og gjør det klart for embedding.

In [ ]:
import numpy as np
import pandas as pd

# Rå data (simulerer hva du ville laste fra en CSV eller database)
raw_data = {
    "id": range(1, 9),
    "tekst": [
        "AFP gir deg rett til å gå av tidlig",
        None,                                          # Mangler tekst
        "  ekstra mellomrom rundt  ",                  # Trenger trimming
        "STORE BOKSTAVER OVERALT",                     # Normalisering
        "Alderspensjon utbetales fra 67 år",
        "Uførepensjon gis ved varig nedsatt arbeidsevne",
        "a",                                           # For kort
        "Barnepensjon sikrer barn under 20 år",
    ],
    "kilde": ["spk", "nav", "spk", "spk", "nav", "spk", "ukjent", "spk"],
    "dato": ["2023-01", "2022-06", "2023-03", None, "2022-11", "2023-05", "2021-08", "2023-02"],
}

df = pd.DataFrame(raw_data)
print(f"Rådata: {len(df)} rader")
print(df)

# --- Rensepipeline ---

# 1. Fjern rader uten tekst
df = df.dropna(subset=["tekst"])

# 2. Trim og normaliser tekst
df["tekst"] = df["tekst"].str.strip().str.lower()

# 3. Fjern for korte tekster (under 10 tegn er meningsløst å embedde)
df = df[df["tekst"].str.len() >= 10]

# 4. Fyll manglende dato
df["dato"] = df["dato"].fillna("ukjent")

# 5. Legg til tekstlengde som ny kolonne (nyttig for chunking-beslutninger)
df["tegn_antall"] = df["tekst"].str.len()

print(f"\nEtter rensing: {len(df)} rader")
print(df[["id", "tekst", "kilde", "tegn_antall"]])

In [ ]:
# Konverter til liste med dicts — formatet de fleste AI-biblioteker forventer
dokumenter = df.to_dict(orient="records")

print("Klar for embedding!")
for dok in dokumenter:
    print(f"  [{dok['kilde']}] {dok['tekst'][:50]}")

---

## Oppsummering

| Konsept | Hva det er | Bruk i AI |
|---------|-----------|----------|
| `np.array` | Effektiv talliste | Embeddings, vektorer, matriser |
| Vektorisering | Matematikk på hele arrays | Cosinus-likhet, normalisering |
| `pd.DataFrame` | Tabelldatastruktur | Laste inn og rense datasett |
| `fillna`, `dropna` | Håndtere manglende data | Alltid nødvendig i virkeligheten |
| `groupby` | Aggregering | Forstå datafordeling |

---

## Hva er neste steg?

**Neste notatbok: `03_ml_concepts_primer.ipynb`**  
Nå som du kan lese og manipulere data, skal vi bygge en konceptuell forståelse av maskinlæring — uten å skrive en eneste treningsfunksjon. Du lærer ordforrådet og ideene som brukes i alle ML/AI-diskusjoner.